# Lab 7: Tavily Web Search — Regulatory Intelligence
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Add real-time web search capability to your agent using the Tavily API. You’ll learn:
- Integrating external APIs as custom function tools
- Using Tavily for structured web search results
- Building a regulatory intelligence agent with current information
- Combining web search with agent reasoning

## Why Web Search?
Documents and databases contain historical data. But insurance regulations change frequently. A web search tool gives your agent access to:
- Latest regulatory updates and compliance requirements
- Industry news and emerging trends
- NAIC model laws and state-specific regulations

> ⚠️ **Prerequisite:** You need a Tavily API key. Get a free one at [tavily.com](https://tavily.com) (1,000 searches/month free).

## Step 1: Import Dependencies

In [1]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import (
    get_clients, MODEL, print_header, print_step, ask_with_functions,
)
from azure.ai.projects.models import PromptAgentDefinition
from tavily import TavilyClient

project_client, openai_client = get_clients()
print("✅ Clients created")

✅ Clients created


## Step 2: Define the Web Search Function

We wrap Tavily’s search API in a function that:
1. Accepts a search query string
2. Returns up to 5 results with title, URL, and content snippet
3. Includes Tavily’s AI-generated answer summary
4. Handles errors gracefully with JSON error responses

In [11]:
tavily_client = None

def web_search(query: str) -> str:
    """Search the web for current information using Tavily API."""
    global tavily_client
    if tavily_client is None:
        api_key = os.environ.get("TAVILY_API_KEY")
        if not api_key:
            return json.dumps({"error": "TAVILY_API_KEY not set"})
        tavily_client = TavilyClient(api_key=api_key)

    try:
        response = tavily_client.search(
            query=query,
            max_results=5,
            include_answer=True,
        )
        results = []
        for r in response.get("results", []):
            results.append({
                "title": r.get("title", ""),
                "url": r.get("url", ""),
                "content": r.get("content", "")[:300],
            })
        return json.dumps({
            "answer": response.get("answer", ""),
            "results": results,
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})

## Step 3: Register as Function Tool

The function schema tells the agent that `web_search` is available and what it does. The agent will call it whenever it needs current information.

In [13]:
FUNCTION_MAP = {"web_search": web_search}

FUNCTION_TOOLS = [
    {
        "type": "function",
        "name": "web_search",
        "description": (
            "Search the web for current information about insurance "
            "regulations, industry news, compliance updates, or any "
            "current events not in the uploaded documents."
            "ALWAYS ABOUT CLAIMS, INSURANCE, REGULATIONS, COMPLIANCE, INDUSTRY NEWS, OR CURRENT EVENTS NOTHING ELSE FROM ANY SOURCES."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query"
                }
            },
            "required": ["query"]
        }
    },
]
print("✅ web_search(query) registered as function tool")

✅ web_search(query) registered as function tool


## Step 4: Check API Key & Create Agent

In [15]:
api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    print("❌ TAVILY_API_KEY not set in .env")
    print("   Get a free key at https://tavily.com")
else:
    print(f"✅ Tavily API key loaded ({api_key[:8]}...)")

agent = project_client.agents.create_version(
    agent_name="smartclaims-regulatory",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims Regulatory Intelligence Agent. "
            "Use the web_search tool to find current insurance "
            "regulations, compliance requirements, and industry "
            "news. Always cite sources with URLs. Present "
            "information in a clear, actionable format."
            "only about claims, insurance, regulations, compliance, industry news, or current events nothing else from any sources."
        ),
        tools=FUNCTION_TOOLS,
    ),
)
print(f"Agent: {agent.name} v{agent.version}")

✅ Tavily API key loaded (tvly-dev...)
Agent: smartclaims-regulatory v3


## Step 5: Query for Regulatory Updates

Let’s test the agent with three regulatory intelligence queries. The agent will search the web in real-time and synthesize the results.

In [ ]:
queries = [
    "What are the latest insurance regulatory changes in the US?",
    "Are there new requirements for AI-powered fraud detection in insurance?",
    "What are the current NAIC model laws regarding data privacy in insurance?",
    "Do you know K21 academy? What do they offer for insurance professionals?",
]

for i, q in enumerate(queries, 1):
    print(f"\n{'─'*60}")
    print(f"❓ Query {i}: {q}")
    conv = openai_client.conversations.create()
    answer = ask_with_functions(
        openai_client, agent, conv.id,
        q, FUNCTION_TOOLS, FUNCTION_MAP,
    )
    print(f"🤖 Answer: {answer}")


────────────────────────────────────────────────────────────
❓ Query 1: Do you know K21 academy? What do they offer for insurance professionals?
    [TOOL CALL] web_search({"query":"K21 academy insurance professionals offerings"})
🤖 Answer: K21 Academy offers various training programs aimed at insurance professionals, focusing on both compliance and practical skills necessary for advancing in the industry. Here are key offerings relevant to insurance:

1. **Compliance Training**:
   - Courses that cover critical compliance areas such as HIPAA, CJIS (Criminal Justice Information Service), and other data protection certifications.
   - For more information on privacy and compliance training, visit [this link](https://k21academy.com/azure-cloud/az-900-privacy-compliance-and-data-protection/).

2. **Professional Certification Prep**:
   - They provide preparation courses for various certifications that can enhance an insurance professional's credentials and job readiness.
   - Information

## Step 6: Clean Up

In [18]:
project_client.agents.delete(agent.name)
print("✅ Agent deleted")

✅ Agent deleted


## ✅ Lab 7 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|-----------------|
| External API integration | Wrap any API as a function tool |
| Tavily search | Structured web search with AI answers |
| Real-time information | Access current data beyond training cutoff |
| Source citation | Agent cites URLs from search results |

---
**Next →** [Lab 8: Deep Observability](lab8_observability.ipynb) — Tracing, metrics & monitoring in Azure AI Foundry